# Notebook 04 â€” Feature Engineering & Final Dataset

**Goal:** Select the final feature set and create the single common dataset
that will be used for all SOH and RUL modelling.

**Input:**  `cleaned_data/processed_dataset.csv`
**Output:** `cleaned_data/featured_dataset.csv`

## Feature Selection Rationale

From EDA we observed:

| Feature | Physical meaning | EDA finding |
|---|---|---|
| `voltage_mean` | Average discharge voltage | Drops as battery ages â€” strong SOH proxy |
| `current_mean` | Average discharge |current| | Roughly constant (controlled experiment) â€” retain as context |
| `temperature_mean` | Average cell temperature | Varies by ambient setting and aging â€” useful for RUL |
| `discharge_duration` | Total discharge time | Decreases with aging â€” correlated with capacity |
| `energy_Wh` | Energy delivered | Highly correlated with capacity â€” useful degradation indicator |
| `ambient_temp_C` | Room temperature | Context variable; influences degradation rate |

**Excluded (data-leakage or redundancy):**
- `capacity` is kept as a mandatory column (SOH target numerator) but must NEVER be
  used as a model input feature for SOH.
- We do not add voltage_min, voltage_max, voltage_std etc. â€” they are redundant
  with voltage_mean for this dataset size.
- Future SOH / EOL cycle / future capacity are NOT computed here.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.join('..', 'src'))
from feature_engineering import FINAL_COLUMNS
from visualization import plot_correlation_heatmap

CLEANED_DIR = os.path.join('..', 'cleaned_data')
# CLEANED_DIR already defined above
PLOT_DIR      = os.path.join('..', 'plots', 'features')
# CLEANED_DIR already created
os.makedirs(PLOT_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(CLEANED_DIR, 'processed_dataset.csv'))
print(f"Input dataset: {df.shape}")
print(f"Columns: {list(df.columns)}")

## 1. Feature Validation

In [ ]:
# Confirm all selected features exist and have acceptable quality
print("FINAL_COLUMNS (from src/feature_engineering.py):")
for c in FINAL_COLUMNS:
    pct_null = df[c].isnull().mean() * 100 if c in df.columns else None
    status = f"{pct_null:.1f}% null" if pct_null is not None else "MISSING"
    print(f"  {c:25s}  {status}")

## 2. Handle Missing Values

In [ ]:
# Check how many rows have any NaN in the final feature columns
feature_only = [c for c in FINAL_COLUMNS if c not in ('battery_id',)]
null_mask = df[feature_only].isnull().any(axis=1)
print(f"Rows with at least one NaN in final columns: {null_mask.sum()}")
if null_mask.sum() > 0:
    print("Columns with NaN:")
    print(df[feature_only].isnull().sum()[df[feature_only].isnull().sum() > 0])
    # Fill energy_Wh NaN with per-battery median (rare edge case)
    if 'energy_Wh' in df.columns and df['energy_Wh'].isnull().sum() > 0:
        medians = df.groupby('battery_id')['energy_Wh'].transform('median')
        df['energy_Wh'] = df['energy_Wh'].fillna(medians)
        print("Filled energy_Wh NaN with per-battery median.")
    null_mask2 = df[feature_only].isnull().any(axis=1)
    print(f"Remaining rows with NaN after fill: {null_mask2.sum()}")
    df = df[~null_mask2].copy()
print(f"Dataset after NaN handling: {df.shape}")

## 3. Build Final Dataset

In [ ]:
# Select only the FINAL_COLUMNS (in specified order)
soh_rul = df[FINAL_COLUMNS].copy()
soh_rul = soh_rul.sort_values(['battery_id', 'cycle']).reset_index(drop=True)

print(f"Final dataset shape: {soh_rul.shape}")
print(f"Columns: {list(soh_rul.columns)}")
print()
print("First 5 rows:")
print(soh_rul.head().to_string())

## 4. Final Quality Check

In [ ]:
print("Missing values in final dataset:")
print(soh_rul.isnull().sum())
print()
print(f"Duplicate (battery_id, cycle) rows: {soh_rul.duplicated(subset=['battery_id','cycle']).sum()}")
print()
print("Descriptive statistics:")
print(soh_rul.drop(columns=['battery_id']).describe().round(4).to_string())

In [ ]:
# Correlation heatmap (feature columns only â€” capacity included to show its relation)
num_cols = [c for c in FINAL_COLUMNS if c != 'battery_id']
fig = plot_correlation_heatmap(soh_rul, num_cols,
    save_path=os.path.join(PLOT_DIR, 'final_feature_correlation.png'))
plt.show()
plt.close(fig)
print("Saved: final_feature_correlation.png")

In [ ]:
# Show correlation with capacity
corr = soh_rul[num_cols].corr()['capacity'].drop('capacity').sort_values(key=abs, ascending=False)
print("Correlations of features with capacity (|r| sorted):")
print(corr.round(4).to_string())
print()
print("Note: high correlation is expected (energy_Wh and discharge_duration")
print("both shrink as capacity shrinks). This confirms these features carry")
print("real degradation information and are NOT leaky (capacity is the target,")
print("not the input).")

## 5. Save Final Dataset

In [ ]:
out_path = os.path.join(CLEANED_DIR, 'featured_dataset.csv')
soh_rul.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Size : {os.path.getsize(out_path) / 1024:.1f} KB")

## Final Dataset Report

In [ ]:
print("=" * 60)
print("  FINAL DATASET REPORT â€” featured_dataset.csv")
print("=" * 60)
print(f"  Rows (one per discharge cycle)  : {len(soh_rul)}")
print(f"  Columns                         : {len(soh_rul.columns)}")
print(f"  Batteries                       : {soh_rul['battery_id'].nunique()}")
print(f"  Total discharge cycles          : {len(soh_rul)}")
print(f"  Missing values                  : {soh_rul.isnull().sum().sum()}")
print(f"  Duplicate rows                  : {soh_rul.duplicated().sum()}")
print()
print("  Mandatory columns:")
for c in ['battery_id', 'cycle', 'capacity']:
    print(f"    {c}")
print()
print("  Selected ML feature columns:")
ml_feats = [c for c in FINAL_COLUMNS if c not in ('battery_id', 'cycle', 'capacity')]
for c in ml_feats:
    corr_val = corr.get(c, float('nan'))
    print(f"    {c:25s}  |r with capacity| = {abs(corr_val):.3f}")
print()
print("  IMPORTANT â€” LEAKAGE RULES for future modelling:")
print("  SOH  : DO NOT use 'capacity' as a model input feature.")
print("         capacity is used to calculate SOH (the target).")
print("  RUL  : DO NOT use EOL_cycle as a model input feature.")
print("         EOL_cycle is used to calculate RUL (the target).")
print()
print("  This dataset is ready for:")
print("    - SOH target generation  (capacity / initial_capacity Ã— 100)")
print("    - RUL target generation  (EOL_cycle - current_cycle)")
print("    - ML training, testing, cross-validation")
print("=" * 60)